In [1]:
import os
import time
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import earthaccess
import h5py

from pyproj import Transformer
from tqdm.auto import tqdm

In [13]:
OUTPUT_DIR = Path(
    "Data/Global/smap"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# SPL3SMP_E V006 DAILY SMAP
# ============================================================

SMAP_DATASET_DIR = Path(
    "Data/Global/pipeline_stage1/"
    "smap_spl3smp_e_v006_parquet_dataset"
)

SMAP_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SMAP_SHORT_NAME = "SPL3SMP_E"
SMAP_VERSION = "006"

SMAP_VARIABLE = "soil_moisture"

SMAP_GROUP_AM = (
    "Soil_Moisture_Retrieval_Data_AM"
)

SMAP_GROUP_PM = (
    "Soil_Moisture_Retrieval_Data_PM"
)

# Physical soil-moisture bounds.
SM_FILL = -9999.0

SM_MIN = None
SM_MAX = None

SMAP_CRS = "EPSG:6933"

# NASA current standard-product reprocessing interval.
SMAP_REPROCESS_START = pd.Timestamp(
    "2026-05-14"
)

SMAP_REPROCESS_END = pd.Timestamp(
    "2026-07-28"
)

print(
    "SMAP dataset directory:",
    SMAP_DATASET_DIR,
)

print(
    "SMAP product:",
    SMAP_SHORT_NAME,
    "V",
    SMAP_VERSION,
)

SMAP dataset directory: Data/Global/pipeline_stage1/smap_spl3smp_e_v006_parquet_dataset
SMAP product: SPL3SMP_E V 006


In [3]:
if "samples" not in globals():
    samples = pd.read_parquet(
        "Data/Global/pipeline_stage1/station_date_candidates.parquet"
    )

if "stations" not in globals():
    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

samples["sample_date"] = pd.to_datetime(
    samples["sample_date"]
).dt.normalize()

print("Samples:", f"{len(samples):,}")
print("Stations:", f"{len(stations):,}")

Samples: 3,514,643
Stations: 2,210


In [4]:
if "station_key" not in samples.columns:
    samples["network"] = samples["network"].astype(str)
    samples["station_id"] = samples["station_id"].astype(str)

    samples["station_key"] = (
        samples["network"]
        + "::"
        + samples["station_id"]
    )

if "station_key" not in stations.columns:
    stations["network"] = stations["network"].astype(str)
    stations["station_id"] = stations["station_id"].astype(str)

    stations["station_key"] = (
        stations["network"]
        + "::"
        + stations["station_id"]
    )

print(
    "Unique station keys:",
    samples["station_key"].nunique()
)

Unique station keys: 2210


In [5]:
auth = earthaccess.login()

print("Earthdata authenticated:", auth.authenticated)

Earthdata authenticated: True


In [6]:
print(
    earthaccess.status()
)

{'Earthdata Login': 'OK', 'Common Metadata Repository': 'OK'}


In [7]:
# ============================================================
# TEST SPL3SMP_E DAILY GRANULE SEARCH
# ============================================================

TEST_DATE = pd.Timestamp(
    "2020-01-15"
).normalize()

TEST_END = (
    TEST_DATE
    + pd.Timedelta(days=1)
)

test_granules = (
    earthaccess.search_data(
        short_name=SMAP_SHORT_NAME,
        version=SMAP_VERSION,
        temporal=(
            TEST_DATE.strftime(
                "%Y-%m-%dT00:00:00"
            ),
            TEST_END.strftime(
                "%Y-%m-%dT00:00:00"
            ),
        ),
        count=-1,
    )
)

print(
    "Test date:",
    TEST_DATE.date(),
)

print(
    "Granules found:",
    len(test_granules),
)

for i, g in enumerate(
    test_granules
):
    print(
        "\nGranule",
        i,
        g,
    )

if not test_granules:
    raise RuntimeError(
        "No SPL3SMP_E V006 granule found "
        f"for {TEST_DATE.date()}."
    )

if len(test_granules) != 1:
    print(
        "\nWARNING:",
        "Expected one daily granule, "
        f"but found {len(test_granules)}."
    )

Test date: 2020-01-15
Granules found: 2

Granule 0 Collection: {'Version': '006', 'ShortName': 'SPL3SMP_E'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -85.04450225830078, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 85.04450225830078}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2020-01-15T23:59:59.999Z', 'BeginningDateTime': '2020-01-15T00:00:00.000Z'}}
Size(MB): 642.6448459625244
Data: ['https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/SMAP/SPL3SMP_E/006/2020/01/15/SMAP_L3_SM_P_E_20200115_R19240_001.h5']

Granule 1 Collection: {'Version': '006', 'ShortName': 'SPL3SMP_E'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -85.04450225830078, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 85.04450225830078}]}}}
Temporal coverage: {'Ran

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:364: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  Size(MB): {self.size()}


In [8]:
# ============================================================
# OPEN TEST DAILY GRANULE
# ============================================================

remote_files = earthaccess.open(
    test_granules[:1],
    show_progress=False,
)

if not remote_files:
    raise RuntimeError(
        "earthaccess.open() returned no file."
    )

f = remote_files[0]

print(
    "Opened test granule:"
)

print(f)

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


Opened test granule:
<File-like object HTTPFileSystem, https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/SMAP/SPL3SMP_E/006/2020/01/15/SMAP_L3_SM_P_E_20200115_R19240_001.h5>


In [12]:
# ============================================================
# INSPECT SPL3SMP_E V006 SOIL-MOISTURE GROUPS ONLY
# ============================================================

with h5py.File(
    f,
    mode="r",
    driver="fileobj",
) as h5:

    print(
        "Top-level groups:"
    )

    print(
        list(h5.keys())
    )

    # --------------------------------------------------------
    # AM group
    # --------------------------------------------------------

    am_group = h5[
        "Soil_Moisture_Retrieval_Data_AM"
    ]

    print(
        "\nAM datasets:"
    )

    print(
        list(am_group.keys())
    )

    # --------------------------------------------------------
    # PM group
    # --------------------------------------------------------

    pm_group = h5[
        "Soil_Moisture_Retrieval_Data_PM"
    ]

    print(
        "\nPM datasets:"
    )

    print(
        list(pm_group.keys())
    )

    # --------------------------------------------------------
    # Expected V006 baseline DCA fields
    # --------------------------------------------------------

    if "soil_moisture_dca" in am_group:
        SMAP_AM_DATASET_PATH = (
            "Soil_Moisture_Retrieval_Data_AM/"
            "soil_moisture_dca"
        )
    else:
        raise RuntimeError(
            "soil_moisture_dca not found in AM group."
        )

    if "soil_moisture_dca_pm" in pm_group:
        SMAP_PM_DATASET_PATH = (
            "Soil_Moisture_Retrieval_Data_PM/"
            "soil_moisture_dca_pm"
        )
    else:
        raise RuntimeError(
            "soil_moisture_dca_pm not found in PM group."
        )

    am_ds = h5[
        SMAP_AM_DATASET_PATH
    ]

    pm_ds = h5[
        SMAP_PM_DATASET_PATH
    ]

    SMAP_GLOBAL_SHAPE = tuple(
        am_ds.shape
    )

    SMAP_PM_SHAPE = tuple(
        pm_ds.shape
    )

    print(
        "\nSelected AM:",
        SMAP_AM_DATASET_PATH,
    )

    print(
        "AM shape:",
        am_ds.shape,
    )

    print(
        "AM chunks:",
        am_ds.chunks,
    )

    print(
        "\nSelected PM:",
        SMAP_PM_DATASET_PATH,
    )

    print(
        "PM shape:",
        pm_ds.shape,
    )

    print(
        "PM chunks:",
        pm_ds.chunks,
    )

    print(
        "\nAM attributes:"
    )

    print(
        dict(am_ds.attrs)
    )

    print(
        "\nPM attributes:"
    )

    print(
        dict(pm_ds.attrs)
    )

if (
    SMAP_GLOBAL_SHAPE
    != SMAP_PM_SHAPE
):
    raise RuntimeError(
        "AM and PM shapes differ: "
        f"{SMAP_GLOBAL_SHAPE} vs "
        f"{SMAP_PM_SHAPE}"
    )

SMAP_NROWS = (
    SMAP_GLOBAL_SHAPE[0]
)

SMAP_NCOLS = (
    SMAP_GLOBAL_SHAPE[1]
)

print(
    "\nGlobal grid:",
    SMAP_GLOBAL_SHAPE,
)

print(
    "Rows:",
    SMAP_NROWS,
)

print(
    "Columns:",
    SMAP_NCOLS,
)

Top-level groups:
['Metadata', 'Soil_Moisture_Retrieval_Data_AM', 'Soil_Moisture_Retrieval_Data_PM', 'Soil_Moisture_Retrieval_Data_Polar_AM', 'Soil_Moisture_Retrieval_Data_Polar_PM']

AM datasets:
['EASE_column_index', 'EASE_row_index', 'albedo', 'albedo_dca', 'albedo_scah', 'albedo_scav', 'boresight_incidence', 'bulk_density', 'clay_fraction', 'freeze_thaw_fraction', 'grid_surface_status', 'latitude', 'latitude_centroid', 'longitude', 'longitude_centroid', 'radar_water_body_fraction', 'retrieval_qual_flag', 'retrieval_qual_flag_dca', 'retrieval_qual_flag_scah', 'retrieval_qual_flag_scav', 'roughness_coefficient', 'roughness_coefficient_dca', 'roughness_coefficient_scah', 'roughness_coefficient_scav', 'soil_moisture', 'soil_moisture_dca', 'soil_moisture_error', 'soil_moisture_scah', 'soil_moisture_scav', 'static_water_body_fraction', 'surface_flag', 'surface_temperature', 'surface_water_fraction_mb_h', 'surface_water_fraction_mb_v', 'tb_3_corrected', 'tb_4_corrected', 'tb_h_corrected',

In [14]:
# ============================================================
# SMAP V006 GRID GEOMETRY DIAGNOSTIC
# ============================================================

print(
    "Product:",
    SMAP_SHORT_NAME,
    "V",
    SMAP_VERSION,
)

print(
    "Grid shape:",
    SMAP_GLOBAL_SHAPE,
)

print(
    "Rows:",
    SMAP_NROWS,
)

print(
    "Columns:",
    SMAP_NCOLS,
)

print(
    "CRS:",
    SMAP_CRS,
)

print(
    "Cell size:",
    SMAP_CELL_SIZE_M
    if "SMAP_CELL_SIZE_M" in globals()
    else "defined in Cell 11",
)

Product: SPL3SMP_E V 006
Grid shape: (1624, 3856)
Rows: 1624
Columns: 3856
CRS: EPSG:6933
Cell size: defined in Cell 11


In [15]:
# ============================================================
# BUILD SMAP STATION -> GRID INDICES
# ============================================================
#
# SPL3SMP_E V006 uses the global 9 km EASE-Grid 2.0
# geometry in EPSG:6933.
#
# Station:
#     lon/lat
#        ->
#     EPSG:6933 metres
#        ->
#     global 9 km grid cell index
# ============================================================

SMAP_CELL_SIZE_M = 9008.05

# Global EASE-Grid 2.0 upper-left outer edge.
SMAP_ULX = -17367530.44
SMAP_ULY = 7314540.83

# ------------------------------------------------------------
# Convert station longitude/latitude to EPSG:6933 metres
# ------------------------------------------------------------

transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x_m, station_y_m = (
    transformer.transform(
        stations["longitude"]
        .to_numpy(dtype=float),

        stations["latitude"]
        .to_numpy(dtype=float),
    )
)

stations_smap = (
    stations.copy()
)

stations_smap["smap_x_m"] = (
    station_x_m
)

stations_smap["smap_y_m"] = (
    station_y_m
)

# ------------------------------------------------------------
# Convert projected coordinates to grid indices
# ------------------------------------------------------------

stations_smap[
    "smap_x_idx"
] = np.floor(
    (
        stations_smap["smap_x_m"]
        - SMAP_ULX
    )
    / SMAP_CELL_SIZE_M
).astype(
    np.int32
)

stations_smap[
    "smap_y_idx"
] = np.floor(
    (
        SMAP_ULY
        - stations_smap["smap_y_m"]
    )
    / SMAP_CELL_SIZE_M
).astype(
    np.int32
)

# ------------------------------------------------------------
# Validate grid indices
# ------------------------------------------------------------

x_idx_valid = (
    (
        stations_smap[
            "smap_x_idx"
        ] >= 0
    )
    &
    (
        stations_smap[
            "smap_x_idx"
        ] < SMAP_NCOLS
    )
)

y_idx_valid = (
    (
        stations_smap[
            "smap_y_idx"
        ] >= 0
    )
    &
    (
        stations_smap[
            "smap_y_idx"
        ] < SMAP_NROWS
    )
)

print(
    "Stations:",
    len(stations_smap),
)

print(
    "X index out of bounds:",
    (~x_idx_valid).sum(),
)

print(
    "Y index out of bounds:",
    (~y_idx_valid).sum(),
)

if (
    not x_idx_valid.all()
    or not y_idx_valid.all()
):

    bad = stations_smap.loc[
        ~(
            x_idx_valid
            & y_idx_valid
        ),
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ],
    ]

    display(
        bad.head(20)
    )

    raise ValueError(
        "Some stations fall outside "
        "the SMAP global grid."
    )

print(
    "All stations have valid "
    "SPL3SMP_E grid indices."
)

print(
    "\nX index range:",
    stations_smap[
        "smap_x_idx"
    ].min(),
    "to",
    stations_smap[
        "smap_x_idx"
    ].max(),
)

print(
    "Y index range:",
    stations_smap[
        "smap_y_idx"
    ].min(),
    "to",
    stations_smap[
        "smap_y_idx"
    ].max(),
)

display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ]
    ].head(10)
)

Stations: 2210
X index out of bounds: 0
Y index out of bounds: 0
All stations have valid SPL3SMP_E grid indices.

X index range: 159 to 3812
Y index range: 40 to 1389


,station_key,latitude,longitude,smap_x_m,smap_y_m,smap_x_idx,smap_y_idx
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,13.53250,2.66040,2.566921e+05,1.710798e+06,1956,622
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,1.649858e+05,1.243671e+06,1946,673
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,1.649858e+05,1.243017e+06,1946,674
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,1.548894e+05,1.237412e+06,1945,674
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,9.74407,1.60580,1.549377e+05,1.237258e+06,1945,674
5,AMMA-CATCH::AMMA-CATCH__Tondikiboro,13.54800,2.69560,2.600884e+05,1.712721e+06,1956,621
6,AMMA-CATCH::AMMA-CATCH__Wankama,13.64560,2.63200,2.539519e+05,1.724833e+06,1956,620
7,ARM::ARM__Anthony,37.21340,-98.09690,-9.465005e+06,4.427857e+06,877,320
8,ARM::ARM__Ashton,37.13300,-97.26600,-9.384835e+06,4.419644e+06,886,321
9,ARM::ARM__Byron,36.88100,-98.28500,-9.483154e+06,4.393847e+06,875,324


In [16]:
# ============================================================
# SMAP GRID GEOMETRY QA
# ============================================================

print(
    "SPL3SMP_E V006 global grid:"
)

print(
    "Rows:",
    SMAP_NROWS,
)

print(
    "Columns:",
    SMAP_NCOLS,
)

print(
    "Expected 9 km global shape:",
    "(1624, 3856)",
)

if (
    SMAP_GLOBAL_SHAPE
    != (1624, 3856)
):
    raise RuntimeError(
        "Unexpected SPL3SMP_E global "
        f"grid shape: {SMAP_GLOBAL_SHAPE}"
    )

print(
    "Grid shape validated."
)

SPL3SMP_E V006 global grid:
Rows: 1624
Columns: 3856
Expected 9 km global shape: (1624, 3856)
Grid shape validated.


In [17]:
"""# Coordinates are in the Metadata group for SPL4SMGP V8
meta_ds = xr.open_dataset(
    f,
    engine="h5netcdf",
    group="Metadata",
    phony_dims="access",
)

x = meta_ds["x"].values
y = meta_ds["y"].values

print("x:", x.shape, x.min(), x.max())
print("y:", y.shape, y.min(), y.max())"""

'# Coordinates are in the Metadata group for SPL4SMGP V8\nmeta_ds = xr.open_dataset(\n    f,\n    engine="h5netcdf",\n    group="Metadata",\n    phony_dims="access",\n)\n\nx = meta_ds["x"].values\ny = meta_ds["y"].values\n\nprint("x:", x.shape, x.min(), x.max())\nprint("y:", y.shape, y.min(), y.max())'

In [18]:
"""transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x, station_y = transformer.transform(
    stations["longitude"].to_numpy(dtype=float),
    stations["latitude"].to_numpy(dtype=float),
)

stations_smap = stations.copy()

stations_smap["smap_x"] = station_x
stations_smap["smap_y"] = station_y"""

'transformer = Transformer.from_crs(\n    "EPSG:4326",\n    SMAP_CRS,\n    always_xy=True,\n)\n\nstation_x, station_y = transformer.transform(\n    stations["longitude"].to_numpy(dtype=float),\n    stations["latitude"].to_numpy(dtype=float),\n)\n\nstations_smap = stations.copy()\n\nstations_smap["smap_x"] = station_x\nstations_smap["smap_y"] = station_y'

In [19]:
"""def nearest_index(values, targets):
    values = np.asarray(values)
    targets = np.asarray(targets)

    ascending = values[0] < values[-1]

    if ascending:
        work = values
    else:
        work = values[::-1]

    idx = np.searchsorted(
        work,
        targets,
        side="left",
    )

    idx = np.clip(
        idx,
        0,
        len(work) - 1,
    )

    left = np.maximum(idx - 1, 0)
    right = idx

    choose_left = (
        np.abs(work[left] - targets)
        <
        np.abs(work[right] - targets)
    )

    idx = np.where(
        choose_left,
        left,
        right,
    )

    if not ascending:
        idx = len(values) - 1 - idx

    return idx.astype(np.int32)"""

'def nearest_index(values, targets):\n    values = np.asarray(values)\n    targets = np.asarray(targets)\n\n    ascending = values[0] < values[-1]\n\n    if ascending:\n        work = values\n    else:\n        work = values[::-1]\n\n    idx = np.searchsorted(\n        work,\n        targets,\n        side="left",\n    )\n\n    idx = np.clip(\n        idx,\n        0,\n        len(work) - 1,\n    )\n\n    left = np.maximum(idx - 1, 0)\n    right = idx\n\n    choose_left = (\n        np.abs(work[left] - targets)\n        <\n        np.abs(work[right] - targets)\n    )\n\n    idx = np.where(\n        choose_left,\n        left,\n        right,\n    )\n\n    if not ascending:\n        idx = len(values) - 1 - idx\n\n    return idx.astype(np.int32)'

In [20]:
# ------------------------------------------------------------
# Build station coordinates in SMAP EPSG:6933
# ------------------------------------------------------------

"""
transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x, station_y = transformer.transform(
    stations["longitude"].to_numpy(dtype=float),
    stations["latitude"].to_numpy(dtype=float),
)

stations_smap = stations.copy()

stations_smap["smap_x"] = station_x
stations_smap["smap_y"] = station_y

print("stations_smap created:")
print("Rows:", len(stations_smap))

display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x",
            "smap_y",
        ]
    ].head()
)"""

'\ntransformer = Transformer.from_crs(\n    "EPSG:4326",\n    SMAP_CRS,\n    always_xy=True,\n)\n\nstation_x, station_y = transformer.transform(\n    stations["longitude"].to_numpy(dtype=float),\n    stations["latitude"].to_numpy(dtype=float),\n)\n\nstations_smap = stations.copy()\n\nstations_smap["smap_x"] = station_x\nstations_smap["smap_y"] = station_y\n\nprint("stations_smap created:")\nprint("Rows:", len(stations_smap))\n\ndisplay(\n    stations_smap[\n        [\n            "station_key",\n            "latitude",\n            "longitude",\n            "smap_x",\n            "smap_y",\n        ]\n    ].head()\n)'

In [21]:
# ============================================================
# SMAP GRID INDEX QA
# ============================================================

# The SMAP x/y variables are pixel indices, NOT metres.
# Therefore we do NOT compare smap_x/smap_y against x/y.

x_idx_valid = (
    (stations_smap["smap_x_idx"] >= 0)
    & (stations_smap["smap_x_idx"] < SMAP_NCOLS)
)

y_idx_valid = (
    (stations_smap["smap_y_idx"] >= 0)
    & (stations_smap["smap_y_idx"] < SMAP_NROWS)
)

print("X index out of bounds:", (~x_idx_valid).sum())
print("Y index out of bounds:", (~y_idx_valid).sum())

if not x_idx_valid.all() or not y_idx_valid.all():
    bad = stations_smap.loc[
        ~(x_idx_valid & y_idx_valid),
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ],
    ]

    display(bad.head(20))

    raise ValueError(
        "Some stations fall outside the SMAP grid."
    )

print("All stations have valid SMAP grid indices.")

print("\nIndex ranges:")
print(
    "X:",
    stations_smap["smap_x_idx"].min(),
    "to",
    stations_smap["smap_x_idx"].max(),
)
print(
    "Y:",
    stations_smap["smap_y_idx"].min(),
    "to",
    stations_smap["smap_y_idx"].max(),
)

X index out of bounds: 0
Y index out of bounds: 0
All stations have valid SMAP grid indices.

Index ranges:
X: 159 to 3812
Y: 40 to 1389


In [22]:
"""check = stations_smap.iloc[:10].copy()

check["grid_x"] = x[
    check["smap_x_idx"].to_numpy()
]

check["grid_y"] = y[
    check["smap_y_idx"].to_numpy()
]

check["dx"] = check["grid_x"] - check["smap_x"]
check["dy"] = check["grid_y"] - check["smap_y"]

display(
    check[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x",
            "smap_y",
            "smap_x_idx",
            "smap_y_idx",
            "dx",
            "dy",
        ]
    ]
)"""

'check = stations_smap.iloc[:10].copy()\n\ncheck["grid_x"] = x[\n    check["smap_x_idx"].to_numpy()\n]\n\ncheck["grid_y"] = y[\n    check["smap_y_idx"].to_numpy()\n]\n\ncheck["dx"] = check["grid_x"] - check["smap_x"]\ncheck["dy"] = check["grid_y"] - check["smap_y"]\n\ndisplay(\n    check[\n        [\n            "station_key",\n            "latitude",\n            "longitude",\n            "smap_x",\n            "smap_y",\n            "smap_x_idx",\n            "smap_y_idx",\n            "dx",\n            "dy",\n        ]\n    ]\n)'

In [23]:
"""with xr.open_dataset(
    f,
    engine="h5netcdf",
    group="Geophysical_Data",
    phony_dims="access",
) as geo:

    sm = geo["sm_surface"]

    print("dims:", sm.dims)
    print("shape:", sm.shape)
    print("attrs:", sm.attrs)
    print("encoding:", sm.encoding)

    raw = sm.values

    finite = np.isfinite(raw)

    print("finite:", finite.sum())
    print("total:", raw.size)

    if finite.any():
        print("min:", np.nanmin(raw))
        print("max:", np.nanmax(raw))
        print("mean:", np.nanmean(raw))"""

'with xr.open_dataset(\n    f,\n    engine="h5netcdf",\n    group="Geophysical_Data",\n    phony_dims="access",\n) as geo:\n\n    sm = geo["sm_surface"]\n\n    print("dims:", sm.dims)\n    print("shape:", sm.shape)\n    print("attrs:", sm.attrs)\n    print("encoding:", sm.encoding)\n\n    raw = sm.values\n\n    finite = np.isfinite(raw)\n\n    print("finite:", finite.sum())\n    print("total:", raw.size)\n\n    if finite.any():\n        print("min:", np.nanmin(raw))\n        print("max:", np.nanmax(raw))\n        print("mean:", np.nanmean(raw))'

In [24]:
"""stations_smap[
    [
        "station_key",
        "latitude",
        "longitude",
        "smap_x",
        "smap_y",
        "smap_x_idx",
        "smap_y_idx",
    ]
].head(20)"""

'stations_smap[\n    [\n        "station_key",\n        "latitude",\n        "longitude",\n        "smap_x",\n        "smap_y",\n        "smap_x_idx",\n        "smap_y_idx",\n    ]\n].head(20)'

In [25]:
# ============================================================
# SMAP STATION -> GRID CELL QA
# ============================================================

stations_smap["smap_cell_x_m"] = (
    SMAP_ULX
    + (
        stations_smap["smap_x_idx"]
        + 0.5
    ) * SMAP_CELL_SIZE_M
)

stations_smap["smap_cell_y_m"] = (
    SMAP_ULY
    - (
        stations_smap["smap_y_idx"]
        + 0.5
    ) * SMAP_CELL_SIZE_M
)

stations_smap["smap_distance_m"] = np.sqrt(
    (
        stations_smap["smap_cell_x_m"]
        - stations_smap["smap_x_m"]
    ) ** 2
    +
    (
        stations_smap["smap_cell_y_m"]
        - stations_smap["smap_y_m"]
    ) ** 2
)

print("Station-to-SMAP-cell distance:")
display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_idx",
            "smap_y_idx",
            "smap_distance_m",
        ]
    ].head(10)
)

display(
    stations_smap["smap_distance_m"].describe()
)

Station-to-SMAP-cell distance:


,station_key,latitude,longitude,smap_x_idx,smap_y_idx,smap_distance_m
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,13.53250,2.66040,1956,622,3767.902492
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,1946,673,4280.447283
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,1946,674,4705.680062
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,1945,674,2992.014891
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,9.74407,1.60580,1945,674,3014.100764
5,AMMA-CATCH::AMMA-CATCH__Tondikiboro,13.54800,2.69560,1956,621,4727.466955
6,AMMA-CATCH::AMMA-CATCH__Wankama,13.64560,2.63200,1956,620,2775.703200
7,ARM::ARM__Anthony,37.21340,-98.09690,877,320,2076.575734
8,ARM::ARM__Ashton,37.13300,-97.26600,886,321,3172.609348
9,ARM::ARM__Byron,36.88100,-98.28500,875,324,3249.884221


count    2210.000000
mean     3409.818882
std      1302.078771
min       103.825157
25%      2486.754698
50%      3490.617712
75%      4461.233783
max      6210.330489
Name: smap_distance_m, dtype: float64

In [26]:
SMAP_STATION_MAP = (
    OUTPUT_DIR
    / "station_smap_grid_mapping.parquet"
)

stations_smap.to_parquet(
    SMAP_STATION_MAP,
    index=False,
)

print("Saved:", SMAP_STATION_MAP)

Saved: Data/Global/smap/station_smap_grid_mapping.parquet


In [30]:
# ============================================================
# SPL3SMP_E V006 DAILY GRANULE READER
# ROW-OPTIMIZED VERSION
# ============================================================

def read_spl3smpe_daily_station_values(
    granule,
    stations_subset,
):
    """
    Read one SPL3SMP_E V006 daily granule.

    Returns:
        am_values : (n_stations,)
        pm_values : (n_stations,)

    The source product contains separate AM and PM
    daily DCA composites.

    The datasets are chunked as:
        (1, 3856)

    Therefore only the unique Y rows required by the
    requested stations are read.
    """

    remote_files = earthaccess.open(
        [granule],
        show_progress=False,
    )

    if not remote_files:
        raise RuntimeError(
            "earthaccess.open() returned no file."
        )

    f = remote_files[0]

    station_y = (
        stations_subset[
            "smap_y_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    station_x = (
        stations_subset[
            "smap_x_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    unique_y = np.unique(
        station_y
    )

    def read_station_values(
        dataset
    ):
        """
        Read only the HDF5 rows required by the stations.
        """

        # ----------------------------------------------------
        # Read only required rows.
        #
        # Dataset chunks are (1, 3856), so this reads exactly
        # the compressed rows needed by the stations.
        # ----------------------------------------------------

        rows = np.asarray(
            dataset[
                unique_y,
                :
            ],
            dtype=np.float32,
        )

        # ----------------------------------------------------
        # Convert station Y coordinates into positions inside
        # the unique-row array.
        # ----------------------------------------------------

        row_positions = np.searchsorted(
            unique_y,
            station_y,
        )

        values = (
            rows[
                row_positions,
                station_x,
            ]
            .astype(
                np.float32
            )
        )

        # ----------------------------------------------------
        # Use the actual metadata stored in the V006 dataset.
        # ----------------------------------------------------

        fill_value = float(
            dataset.attrs.get(
                "_FillValue",
                SM_FILL,
            )
        )

        valid_min = float(
            dataset.attrs[
                "valid_min"
            ]
        )

        valid_max = float(
            dataset.attrs[
                "valid_max"
            ]
        )

        # ----------------------------------------------------
        # Fill value -> NaN
        # ----------------------------------------------------

        values[
            values == fill_value
        ] = np.nan

        # ----------------------------------------------------
        # Product-defined valid range
        # ----------------------------------------------------

        values[
            (values < valid_min)
            |
            (values > valid_max)
        ] = np.nan

        return values

    # ========================================================
    # OPEN HDF5
    # ========================================================

    with h5py.File(
        f,
        mode="r",
        driver="fileobj",
    ) as h5:

        # ----------------------------------------------------
        # AM
        # ----------------------------------------------------

        am_dataset = h5[
            SMAP_AM_DATASET_PATH
        ]

        am_values = (
            read_station_values(
                am_dataset
            )
        )

        # ----------------------------------------------------
        # PM
        # ----------------------------------------------------

        pm_dataset = h5[
            SMAP_PM_DATASET_PATH
        ]

        pm_values = (
            read_station_values(
                pm_dataset
            )
        )

    return (
        am_values,
        pm_values,
    )


print(
    "SPL3SMP_E V006 row-optimized daily reader ready."
)

SPL3SMP_E V006 row-optimized daily reader ready.


In [31]:
# ============================================================
# BENCHMARK OPTIMIZED SPL3SMP_E DAILY EXTRACTION
# ============================================================

benchmark_stations = (
    stations_smap
)

print(
    "Stations:",
    len(benchmark_stations),
)

print(
    "Unique Y rows:",
    benchmark_stations[
        "smap_y_idx"
    ].nunique(),
)

print(
    "Total grid rows:",
    SMAP_NROWS,
)

print(
    "Fraction of rows needed:",
    (
        benchmark_stations[
            "smap_y_idx"
        ].nunique()
        /
        SMAP_NROWS
    )
)

t0 = time.perf_counter()

test_am, test_pm = (
    read_spl3smpe_daily_station_values(
        test_granules[0],
        benchmark_stations,
    )
)

elapsed = (
    time.perf_counter()
    - t0
)

test_result = pd.DataFrame({
    "station_key":
        benchmark_stations[
            "station_key"
        ].to_numpy(),

    "SMAP_AM":
        test_am,

    "SMAP_PM":
        test_pm,
})

test_result["SMAP"] = (
    test_result[
        [
            "SMAP_AM",
            "SMAP_PM",
        ]
    ]
    .mean(
        axis=1,
        skipna=True,
    )
)

test_result[
    "SMAP_n_obs"
] = (
    test_result[
        [
            "SMAP_AM",
            "SMAP_PM",
        ]
    ]
    .notna()
    .sum(
        axis=1
    )
    .astype(
        np.int8
    )
)

print(
    "\nElapsed:",
    f"{elapsed:.2f} s",
)

print(
    "Shape:",
    test_result.shape,
)

print(
    "\nAM valid:",
    test_result[
        "SMAP_AM"
    ].notna().sum(),
    "/",
    len(test_result),
)

print(
    "PM valid:",
    test_result[
        "SMAP_PM"
    ].notna().sum(),
    "/",
    len(test_result),
)

print(
    "Daily valid:",
    test_result[
        "SMAP"
    ].notna().sum(),
    "/",
    len(test_result),
)

print(
    "Mean observations:",
    test_result[
        "SMAP_n_obs"
    ].mean(),
)

display(
    test_result.head()
)

Stations: 2210
Unique Y rows: 397
Total grid rows: 1624
Fraction of rows needed: 0.24445812807881773



Elapsed: 106.78 s
Shape: (2210, 5)

AM valid: 503 / 2210
PM valid: 712 / 2210
Daily valid: 1011 / 2210
Mean observations: 0.5497737556561086


,station_key,SMAP_AM,SMAP_PM,SMAP,SMAP_n_obs
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0.034559,0.034559,1
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,NaN,0.059775,0.059775,1
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,NaN,0.066271,0.066271,1
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,NaN,0.061746,0.061746,1
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,NaN,0.061746,0.061746,1


In [32]:
# ============================================================
# SPL3SMP_E V006 COVERAGE DIAGNOSTIC
# ============================================================

station_y = (
    stations_smap["smap_y_idx"]
    .to_numpy(dtype=np.int32)
)

station_x = (
    stations_smap["smap_x_idx"]
    .to_numpy(dtype=np.int32)
)

unique_y = np.unique(station_y)

print("Stations:", len(stations_smap))
print("Unique Y rows:", len(unique_y))
print("Total grid rows:", SMAP_NROWS)
print(
    "Fraction of grid rows needed:",
    f"{len(unique_y) / SMAP_NROWS:.3%}"
)

print(
    "\nY range:",
    station_y.min(),
    "to",
    station_y.max(),
)

print(
    "X range:",
    station_x.min(),
    "to",
    station_x.max(),
)

print("\nBenchmark coverage:")
print(
    "AM:",
    f"{test_result['SMAP_AM'].notna().sum()} / {len(test_result)}"
)
print(
    "PM:",
    f"{test_result['SMAP_PM'].notna().sum()} / {len(test_result)}"
)
print(
    "Daily:",
    f"{test_result['SMAP'].notna().sum()} / {len(test_result)}"
)

print("\nStations with:")
print(
    "  AM only:",
    (
        test_result["SMAP_AM"].notna()
        & test_result["SMAP_PM"].isna()
    ).sum()
)
print(
    "  PM only:",
    (
        test_result["SMAP_AM"].isna()
        & test_result["SMAP_PM"].notna()
    ).sum()
)
print(
    "  Both:",
    (
        test_result["SMAP_AM"].notna()
        & test_result["SMAP_PM"].notna()
    ).sum()
)
print(
    "  Neither:",
    (
        test_result["SMAP_AM"].isna()
        & test_result["SMAP_PM"].isna()
    ).sum()
)

Stations: 2210
Unique Y rows: 397
Total grid rows: 1624
Fraction of grid rows needed: 24.446%

Y range: 40 to 1389
X range: 159 to 3812

Benchmark coverage:
AM: 503 / 2210
PM: 712 / 2210
Daily: 1011 / 2210

Stations with:
  AM only: 299
  PM only: 508
  Both: 204
  Neither: 1199


In [ ]:
# ============================================================
# VALIDATE SPL3SMP_E DAILY TEST
# ============================================================

valid_am = (
    test_result[
        "SMAP_AM"
    ].notna().sum()
)

valid_pm = (
    test_result[
        "SMAP_PM"
    ].notna().sum()
)

valid_daily = (
    test_result[
        "SMAP"
    ].notna().sum()
)

print(
    "AM valid:",
    valid_am,
    "/",
    len(test_result),
)

print(
    "PM valid:",
    valid_pm,
    "/",
    len(test_result),
)

print(
    "Daily valid:",
    valid_daily,
    "/",
    len(test_result),
)

display(
    test_result[
        [
            "SMAP_AM",
            "SMAP_PM",
            "SMAP",
        ]
    ].describe()
)

if valid_daily == 0:
    raise RuntimeError(
        "SPL3SMP_E daily extraction returned "
        "zero valid station values."
    )

print(
    "\nSPL3SMP_E daily product test passed."
)

Valid: 2210 / 2210
Valid percentage: 100.00%


count    2210.000000
mean        0.232079
std         0.123514
min         0.021884
25%         0.138911
50%         0.213832
75%         0.293025
max         0.797541
Name: SMAP_3h, dtype: float64

SMAP single-granule test passed.


In [ ]:
"""def get_smap_granules_for_date(date):
    
    Search all SPL4SMGP V008 granules for one UTC calendar day.
    

    date = pd.Timestamp(date).normalize()

    next_day = date + pd.Timedelta(days=1)

    return earthaccess.search_data(
        short_name=SMAP_SHORT_NAME,
        version=SMAP_VERSION,
        temporal=(
            date.strftime("%Y-%m-%dT00:00:00"),
            next_day.strftime("%Y-%m-%dT00:00:00"),
        ),
        count=-1,
    )"""

'def get_smap_granules_for_date(date):\n\n    Search all SPL4SMGP V008 granules for one UTC calendar day.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    next_day = date + pd.Timedelta(days=1)\n\n    return earthaccess.search_data(\n        short_name=SMAP_SHORT_NAME,\n        version=SMAP_VERSION,\n        temporal=(\n            date.strftime("%Y-%m-%dT00:00:00"),\n            next_day.strftime("%Y-%m-%dT00:00:00"),\n        ),\n        count=-1,\n    )'

In [ ]:
"""def extract_smap_day(
    date,
    stations_smap,
):
    
    Extract all available 3-hourly SMAP observations for one UTC day,
    then average them to a daily surface-soil-moisture value per station.
    

    date = pd.Timestamp(date).normalize()

    granules = get_smap_granules_for_date(date)

    if not granules:
        return pd.DataFrame({
            "station_key": stations_smap["station_key"],
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    parts = []

    for granule in granules:

        try:
            values = extract_granule_station_values(
                granule,
                stations_smap,
            )

            parts.append(
                values["SMAP_3h"].to_numpy()
            )

        except Exception as exc:
            print(
                f"Warning: failed granule "
                f"{granule}: {exc}"
            )

    if not parts:
        return pd.DataFrame({
            "station_key": stations_smap["station_key"],
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    values = np.vstack(parts)

    valid = np.isfinite(values)

    count = valid.sum(axis=0)

    values_for_mean = np.where(
        valid,
        values,
        np.nan,
    )

    daily_mean = np.nanmean(
        values_for_mean,
        axis=0,
    )

    result = pd.DataFrame({
        "station_key": stations_smap["station_key"].to_numpy(),
        "sample_date": date,
        "SMAP": daily_mean.astype(np.float32),
        "SMAP_n_obs": count.astype(np.int8),
    })

    return result"""

'def extract_smap_day(\n    date,\n    stations_smap,\n):\n\n    Extract all available 3-hourly SMAP observations for one UTC day,\n    then average them to a daily surface-soil-moisture value per station.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    granules = get_smap_granules_for_date(date)\n\n    if not granules:\n        return pd.DataFrame({\n            "station_key": stations_smap["station_key"],\n            "sample_date": date,\n            "SMAP": np.nan,\n            "SMAP_n_obs": 0,\n        })\n\n    parts = []\n\n    for granule in granules:\n\n        try:\n            values = extract_granule_station_values(\n                granule,\n                stations_smap,\n            )\n\n            parts.append(\n                values["SMAP_3h"].to_numpy()\n            )\n\n        except Exception as exc:\n            print(\n                f"Warning: failed granule "\n                f"{granule}: {exc}"\n            )\n\n    if not parts:\n        return pd.Da

In [ ]:
sample_dates = (
    samples["sample_date"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(
    "Unique ISMN dates:",
    len(sample_dates)
)

print(
    "Range:",
    sample_dates[0],
    "→",
    sample_dates[-1],
)

Unique ISMN dates: 4204
Range: 2015-04-01 00:00:00 → 2026-10-03 00:00:00


In [ ]:
date_to_station_keys = {}

for date, group in samples.groupby("sample_date"):
    date_to_station_keys[
        pd.Timestamp(date)
    ] = set(
        group["station_key"]
        .astype(str)
    )

print(
    "Dates:",
    len(date_to_station_keys)
)

Dates: 4204


In [ ]:
"""station_lookup = stations_smap.set_index(
    "station_key"
)"""

'station_lookup = stations_smap.set_index(\n    "station_key"\n)'

In [ ]:
"""def extract_smap_day_for_samples(
    date,
    stations_smap,
    requested_station_keys,
):
   
    Extract daily SMAP only for stations that actually have
    an ISMN sample on this date.
   

    date = pd.Timestamp(date).normalize()

    requested_station_keys = list(
        requested_station_keys
    )

    station_subset = (
        stations_smap[
            stations_smap["station_key"]
            .isin(requested_station_keys)
        ]
        .reset_index(drop=True)
    )

    granules = get_smap_granules_for_date(date)

    if not granules:
        return pd.DataFrame({
            "station_key": requested_station_keys,
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    parts = []

    for granule in granules:
        try:
            remote_files = earthaccess.open(
                [granule],
                show_progress=False,
            )

            f = remote_files[0]

            with xr.open_dataset(
                f,
                engine="h5netcdf",
                group="Geophysical_Data",
                phony_dims="access",
            ) as geo:

                sm = geo["sm_surface"]

                x = geo["x"].values
                y = geo["y"].values

                x_dim = next(
                    d for d in sm.dims
                    if sm.sizes[d] == len(x)
                )

                y_dim = next(
                    d for d in sm.dims
                    if sm.sizes[d] == len(y)
                )

                y_idx = xr.DataArray(
                    station_subset["smap_y_idx"].to_numpy(),
                    dims="station_index",
                )

                x_idx = xr.DataArray(
                    station_subset["smap_x_idx"].to_numpy(),
                    dims="station_index",
                )

                values = (
                    sm.isel({
                        y_dim: y_idx,
                        x_dim: x_idx,
                    })
                    .values
                    .astype(np.float32)
                )

                values[~np.isfinite(values)] = np.nan

                values[
                    (values < SM_MIN)
                    | (values > SM_MAX)
                ] = np.nan

                parts.append(values)

        except Exception as exc:
            print(
                f"Warning: failed granule: {exc}"
            )

    if not parts:
        return pd.DataFrame({
            "station_key": requested_station_keys,
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    array = np.vstack(parts)

    valid = np.isfinite(array)

    counts = valid.sum(axis=0)

    daily = np.nanmean(
        np.where(
            valid,
            array,
            np.nan,
        ),
        axis=0,
    )

    result = pd.DataFrame({
        "station_key": station_subset["station_key"],
        "sample_date": date,
        "SMAP": daily.astype(np.float32),
        "SMAP_n_obs": counts.astype(np.int8),
    })

    return result"""

'def extract_smap_day_for_samples(\n    date,\n    stations_smap,\n    requested_station_keys,\n):\n\n    Extract daily SMAP only for stations that actually have\n    an ISMN sample on this date.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    requested_station_keys = list(\n        requested_station_keys\n    )\n\n    station_subset = (\n        stations_smap[\n            stations_smap["station_key"]\n            .isin(requested_station_keys)\n        ]\n        .reset_index(drop=True)\n    )\n\n    granules = get_smap_granules_for_date(date)\n\n    if not granules:\n        return pd.DataFrame({\n            "station_key": requested_station_keys,\n            "sample_date": date,\n            "SMAP": np.nan,\n            "SMAP_n_obs": 0,\n        })\n\n    parts = []\n\n    for granule in granules:\n        try:\n            remote_files = earthaccess.open(\n                [granule],\n                show_progress=False,\n            )\n\n            f = remote_files[0]\n\n  

In [ ]:
"""SMAP_OUTPUT = (
    OUTPUT_DIR
    / "smap_by_sample.parquet"
)

print(SMAP_OUTPUT)"""

'SMAP_OUTPUT = (\n    OUTPUT_DIR\n    / "smap_by_sample.parquet"\n)\n\nprint(SMAP_OUTPUT)'

In [ ]:
# ============================================================
# SAMPLE KEYS FOR SMAP JOIN
# ============================================================

sample_columns = [
    "sample_id",
    "station_key",
    "station_id",
    "network",
    "sample_date",
]

sample_keys = (
    samples[sample_columns]
    .copy()
)

sample_keys["sample_date"] = (
    pd.to_datetime(
        sample_keys["sample_date"]
    )
    .dt.normalize()
)

sample_keys["station_key"] = (
    sample_keys["station_key"]
    .astype(str)
)

# sample_id must uniquely identify each final sample row.
duplicate_sample_ids = (
    sample_keys["sample_id"]
    .duplicated(keep=False)
)

if duplicate_sample_ids.any():

    display(
        sample_keys.loc[
            duplicate_sample_ids
        ].head(20)
    )

    raise ValueError(
        "Duplicate sample_id values found."
    )

print(
    "Sample rows:",
    f"{len(sample_keys):,}"
)

print(
    "Unique station/date pairs:",
    f"{sample_keys[['station_key', 'sample_date']]}"
    ".drop_duplicates().shape[0]:,}"
)

display(
    sample_keys.head()
)

Sample rows: 3,514,643
Unique station/date pairs:                                  station_key sample_date
0        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-01
1        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-02
2        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-03
3        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-04
4        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-05
...                                      ...         ...
3514638            iRON::iRON__Spring_Valley  2025-09-21
3514639            iRON::iRON__Spring_Valley  2025-09-22
3514640            iRON::iRON__Spring_Valley  2025-09-23
3514641            iRON::iRON__Spring_Valley  2025-09-24
3514642            iRON::iRON__Spring_Valley  2025-09-25

[3514643 rows x 2 columns].drop_duplicates().shape[0]:,}


,sample_id,station_key,station_id,network,sample_date
0,0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-01
1,1,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-02
2,2,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-03
3,3,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-04
4,4,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-05


In [ ]:
# ============================================================
# BULK SPL3SMP_E V006 SEARCH
# + DAILY PRODUCTION EXTRACTOR
# ============================================================

print(
    "Performing one bulk CMR search "
    "for SPL3SMP_E V006..."
)

search_start = (
    sample_dates[0]
    .strftime(
        "%Y-%m-%dT00:00:00"
    )
)

search_end = (
    sample_dates[-1]
    + pd.Timedelta(days=1)
).strftime(
    "%Y-%m-%dT00:00:00"
)

all_granules = (
    earthaccess.search_data(
        short_name=SMAP_SHORT_NAME,
        version=SMAP_VERSION,
        temporal=(
            search_start,
            search_end,
        ),
        count=-1,
    )
)

print(
    "Granules found:",
    len(all_granules),
)

# ------------------------------------------------------------
# Keep only ISMN sample dates.
# ------------------------------------------------------------

requested_dates = {
    pd.Timestamp(d).normalize()
    for d in sample_dates
}

granules_by_date = {}

for g in all_granules:

    dt_str = (
        g["umm"]
        ["TemporalExtent"]
        ["RangeDateTime"]
        ["BeginningDateTime"]
    )

    dt = (
        pd.to_datetime(
            dt_str,
            utc=True,
        )
        .tz_localize(None)
        .normalize()
    )

    if dt in requested_dates:

        granules_by_date.setdefault(
            dt,
            []
        ).append(g)

print(
    "Sample dates with SMAP granules:",
    len(granules_by_date),
)

print(
    "Sample dates without SMAP granules:",
    len(
        requested_dates
        - set(
            granules_by_date.keys()
        )
    ),
)

# ------------------------------------------------------------
# Daily-product QA:
# normally exactly one granule per date.
# ------------------------------------------------------------

multi_granule_dates = {
    d: gs
    for d, gs in
    granules_by_date.items()
    if len(gs) != 1
}

print(
    "Dates with != 1 granule:",
    len(multi_granule_dates),
)

if multi_granule_dates:

    first_date = next(
        iter(
            multi_granule_dates
        )
    )

    print(
        "Example:",
        first_date.date(),
        "->",
        len(
            multi_granule_dates[
                first_date
            ]
        ),
        "granules",
    )

# ------------------------------------------------------------
# Station lookup.
# ------------------------------------------------------------

stations_smap_lookup = (
    stations_smap
    .set_index(
        "station_key",
        verify_integrity=True,
    )
)

# ------------------------------------------------------------
# Fast sample metadata lookup.
#
# This avoids merging the entire 3.5M-row sample_keys
# table for every date.
# ------------------------------------------------------------

sample_meta_lookup = (
    sample_keys[
        [
            "sample_id",
            "station_id",
            "network",
        ]
    ]
    .set_index(
        [
            "station_key",
            "sample_date",
        ]
    )
)

# ------------------------------------------------------------
# Production daily extractor.
# ------------------------------------------------------------

def extract_smap_day_for_samples(
    date,
    stations_smap_lookup,
    requested_station_keys,
):
    """
    Extract one SPL3SMP_E V006 daily value
    per requested station.

    AM and PM daily composites are averaged
    when available.

    SMAP_n_obs:
        0 = neither AM nor PM valid
        1 = one valid composite
        2 = both valid
    """

    date = pd.Timestamp(
        date
    ).normalize()

    requested_station_keys = sorted(
        map(
            str,
            requested_station_keys,
        )
    )

    # --------------------------------------------------------
    # Resolve stations.
    # --------------------------------------------------------

    missing_keys = (
        pd.Index(
            requested_station_keys
        )
        .difference(
            stations_smap_lookup.index
        )
    )

    if len(missing_keys) > 0:

        raise KeyError(
            "Stations missing from "
            "stations_smap: "
            + ", ".join(
                map(
                    str,
                    missing_keys[:10],
                )
            )
        )

    station_subset = (
        stations_smap_lookup
        .loc[
            requested_station_keys
        ]
        .reset_index()
    )

    # --------------------------------------------------------
    # Find the daily granule.
    # --------------------------------------------------------

    granules = (
        granules_by_date.get(
            date,
            [],
        )
    )

    if not granules:

        return pd.DataFrame({
            "station_key":
                station_subset[
                    "station_key"
                ].to_numpy(),

            "sample_date":
                date,

            "SMAP":
                np.nan,

            "SMAP_n_obs":
                np.zeros(
                    len(station_subset),
                    dtype=np.int8,
                ),
        })

    if len(granules) != 1:

        raise RuntimeError(
            f"Expected exactly one "
            f"SPL3SMP_E granule for "
            f"{date.date()}, "
            f"got {len(granules)}."
        )

    # --------------------------------------------------------
    # Read AM + PM.
    # --------------------------------------------------------

    am_values, pm_values = (
        read_spl3smpe_daily_station_values(
            granules[0],
            station_subset,
        )
    )

    array = np.vstack(
        [
            am_values,
            pm_values,
        ]
    )

    valid = np.isfinite(
        array
    )

    counts = (
        valid
        .sum(axis=0)
        .astype(np.int8)
    )

    sums = np.nansum(
        array,
        axis=0,
        dtype=np.float32,
    )

    daily = np.full(
        counts.shape,
        np.nan,
        dtype=np.float32,
    )

    has_values = (
        counts > 0
    )

    daily[
        has_values
    ] = (
        sums[has_values]
        /
        counts[has_values]
    )

    return pd.DataFrame({
        "station_key":
            station_subset[
                "station_key"
            ].to_numpy(),

        "sample_date":
            date,

        "SMAP":
            daily,

        "SMAP_n_obs":
            counts,
    })


print(
    "SPL3SMP_E V006 production extractor ready."
)

Performing one bulk CMR search for SMAP granules...


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Granules found: 33634
Sample dates with SMAP granules: 4204
Sample dates without SMAP granules: 0
Sample dates with SMAP granules: 4204
Sample dates without SMAP granules: 0
Production SMAP extractor ready.
Production SMAP extractor ready.
Production SMAP extractor ready.


In [ ]:
# ============================================================
# SPL3SMP_E V006
# Output directory is defined in Cell 2.
# ============================================================

print(
    "Using SMAP output directory:",
    SMAP_DATASET_DIR,
)

'# Replace SMAP_OUTPUT with a directory path\nSMAP_DATASET_DIR = "Data/Global/pipeline_stage1/smap_parquet_dataset"\nos.makedirs(SMAP_DATASET_DIR, exist_ok=True)'

In [ ]:
# ============================================================
# OLD SPL4SMGP BENCHMARK RETIRED
# ============================================================

print(
    "Cell 36 retired."
)

print(
    "SPL3SMP_E daily benchmark is in "
    "Cells 23-25."
)

Benchmark date: 2015-04-01
Testing granule 2/8


earthaccess.open(): 5.47 s

DIRECT PATH RESULT
h5py.File(): 13.09 s
Direct dataset lookup: 35.95 s
Array read: 8.81 s
Total after earthaccess.open(): 52.39 s
Shape: (1624, 3856)
Finite values: 6262144


In [ ]:
from tqdm.std import tqdm

errors = []

for date in tqdm(
    sample_dates,
    desc="Extracting daily SMAP",
):

    date = pd.Timestamp(
        date
    ).normalize()

    date_str = (
        date.strftime(
            "%Y-%m-%d"
        )
    )

    out_file = (
        SMAP_DATASET_DIR
        / f"smap_{date_str}.parquet"
    )

    # --------------------------------------------------------
    # Completed date -> skip.
    # --------------------------------------------------------

    if out_file.exists():
        continue

    needed_stations = (
        date_to_station_keys[
            date
        ]
    )

    tmp_file = (
        SMAP_DATASET_DIR
        / f".smap_{date_str}.parquet.tmp"
    )

    try:

        # ----------------------------------------------------
        # Extract daily SMAP.
        # ----------------------------------------------------

        daily = (
            extract_smap_day_for_samples(
                date,
                stations_smap_lookup,
                needed_stations,
            )
        )

        daily = daily[
            [
                "station_key",
                "sample_date",
                "SMAP",
                "SMAP_n_obs",
            ]
        ]

        # ----------------------------------------------------
        # Row-count validation BEFORE metadata join.
        # ----------------------------------------------------

        expected_rows = len(
            needed_stations
        )

        if len(daily) != expected_rows:

            raise RuntimeError(
                f"SMAP row-count mismatch "
                f"for {date.date()}: "
                f"expected {expected_rows}, "
                f"got {len(daily)}"
            )

        # ----------------------------------------------------
        # Attach sample metadata using a small indexed lookup.
        # ----------------------------------------------------

        daily = (
            daily
            .set_index(
                [
                    "station_key",
                    "sample_date",
                ]
            )
            .join(
                sample_meta_lookup,
                how="left",
                validate="one_to_one",
            )
            .reset_index()
        )

        # ----------------------------------------------------
        # Verify metadata join.
        # ----------------------------------------------------

        if daily[
            "sample_id"
        ].isna().any():

            missing = int(
                daily[
                    "sample_id"
                ].isna().sum()
            )

            raise RuntimeError(
                f"{missing} SMAP rows could not "
                f"be matched to sample_id for "
                f"{date.date()}."
            )

        # ----------------------------------------------------
        # Final column order.
        # ----------------------------------------------------

        daily = daily[
            [
                "sample_id",
                "station_key",
                "station_id",
                "network",
                "sample_date",
                "SMAP",
                "SMAP_n_obs",
            ]
        ]

        # ----------------------------------------------------
        # Write temporary file.
        # ----------------------------------------------------

        if tmp_file.exists():
            tmp_file.unlink()

        daily.to_parquet(
            tmp_file,
            index=False,
            compression="zstd",
        )

        # ----------------------------------------------------
        # Atomic checkpoint.
        # ----------------------------------------------------

        os.replace(
            tmp_file,
            out_file,
        )

    except Exception as exc:

        if tmp_file.exists():
            tmp_file.unlink()

        errors.append({
            "sample_date": date,
            "error": repr(exc),
        })

        print(
            f"\nFAILED {date.date()}: {exc}"
        )

        continue


print(
    "\nExtraction loop finished."
)

if errors:

    error_df = pd.DataFrame(
        errors
    )

    error_path = (
        SMAP_DATASET_DIR
        / "smap_extraction_errors.csv"
    )

    error_df.to_csv(
        error_path,
        index=False,
    )

    print(
        f"Failed dates: {len(error_df):,}"
    )

    print(
        "Error log:",
        error_path,
    )

else:

    print(
        "No extraction errors."
    )

Extracting SMAP:   0%|          | 0/4204 [00:00<?, ?it/s]/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)
Extracting SMAP:   0%|          | 0/4204 [00:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# LOAD AND VALIDATE COMPLETED SMAP DATASET
# ============================================================

expected_sample_ids = set(
    sample_keys["sample_id"]
)

smap_final = pd.read_parquet(
    SMAP_DATASET_DIR
)

print(
    "SMAP rows:",
    f"{len(smap_final):,}"
)

print(
    "Unique samples:",
    smap_final["sample_id"].nunique()
)

# Every ISMN sample must appear exactly once.
if (
    len(smap_final)
    != len(sample_keys)
):
    raise RuntimeError(
        f"SMAP row count mismatch: "
        f"expected {len(sample_keys):,}, "
        f"got {len(smap_final):,}"
    )

if (
    smap_final["sample_id"].nunique()
    != len(sample_keys)
):
    raise RuntimeError(
        "SMAP output does not contain "
        "exactly one row per sample_id."
    )

missing_sample_ids = (
    expected_sample_ids
    - set(smap_final["sample_id"])
)

if missing_sample_ids:
    raise RuntimeError(
        f"SMAP output is missing "
        f"{len(missing_sample_ids):,} sample IDs."
    )

display(
    smap_final.head()
)

SMAP rows: 3,514,643
Unique samples: 3514643


,sample_id,station_key,station_id,network,sample_date,SMAP,SMAP_n_obs
0,749012,PBO_H2O::PBO_H2O__PARKINSONF,PBO_H2O__PARKINSONF,PBO_H2O,2015-04-01,NaN,0
1,122313,COSMOS-UK::COSMOS-UK__Bickley_Hall,COSMOS-UK__Bickley_Hall,COSMOS-UK,2015-04-01,NaN,0
2,1267880,SCAN::SCAN__Lovelock_NNR,SCAN__Lovelock_NNR,SCAN,2015-04-01,NaN,0
3,1405838,SCAN::SCAN__Princeton_#1,SCAN__Princeton_#1,SCAN,2015-04-01,NaN,0
4,773640,PBO_H2O::PBO_H2O__Upington,PBO_H2O__Upington,PBO_H2O,2015-04-01,NaN,0


In [ ]:
coverage = {
    "total_rows": len(smap_final),

    "nonmissing_smap": int(
        smap_final["SMAP"].notna().sum()
    ),

    "missing_smap": int(
        smap_final["SMAP"].isna().sum()
    ),

    "missing_percent": float(
        smap_final["SMAP"].isna().mean() * 100
    ),

    "mean_observations_per_day": float(
        smap_final["SMAP_n_obs"].mean()
    ),
}

coverage

{'total_rows': 3514643,
 'nonmissing_smap': 0,
 'missing_smap': 3514643,
 'missing_percent': 100.0,
 'mean_observations_per_day': 0.0}

In [ ]:
print(
    smap_final["SMAP"]
    .describe()
)
invalid = (
    smap_final["SMAP"].notna()
    &
    ~smap_final["SMAP"].between(
        SM_MIN,
        SM_MAX,
    )
)

print(
    "Invalid SMAP values:",
    invalid.sum()
)

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: SMAP, dtype: float64
Invalid SMAP values: 0


In [ ]:
smap_final["SMAP_reprocessing_flag"] = (
    smap_final["sample_date"].between(
        SMAP_REPROCESS_START,
        SMAP_REPROCESS_END,
        inclusive="both",
    )
)

print(
    smap_final[
        "SMAP_reprocessing_flag"
    ].value_counts()
)

SMAP_reprocessing_flag
False    3451396
True       63247
Name: count, dtype: int64


In [ ]:
master_with_smap = samples.merge(
    smap_final[
        [
            "sample_id",
            "SMAP",
            "SMAP_n_obs",
            "SMAP_reprocessing_flag",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

print(
    "Master rows:",
    f"{len(master_with_smap):,}"
)

display(
    master_with_smap.head()
)

Master rows: 3,514,643


,sample_id,station_id,sample_date,network,network_name,station,latitude,longitude,elevation_m,climate_zone,source_row_count,min_depth_from_m,max_depth_to_m,sensors,depth_classes,station_key,SMAP,SMAP_n_obs,SMAP_reprocessing_flag
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False


: 

In [ ]:
MASTER_SMAP_OUTPUT = (
    OUTPUT_DIR
    / "global_ismn_smap_index.parquet"
)

master_with_smap.to_parquet(
    MASTER_SMAP_OUTPUT,
    index=False,
)

print(
    "Saved:",
    MASTER_SMAP_OUTPUT
)